# Proceso ETL para la Construcción de una Bodega de Datos de Mensajería

Este cuaderno contiene el desarrollo completo del proceso **ETL (Extracción, Transformación y Carga)** utilizado para construir una **bodega de datos** a partir de una base de datos transaccional de un sistema de mensajería.

A lo largo del notebook se realizan los siguientes pasos:

- **Conexión segura** a las bases de datos mediante variables de entorno.
- **Extracción** de datos relevantes desde la base de datos de origen.
- **Transformación** de los datos para la construcción de:
  - Tablas de **dimensiones**: cliente, mensajero, sede, fecha y hora.
  - Tablas de **hechos**: servicios de mensajería y novedades durante el servicio.
- **Carga** de los datos transformados en la bodega de datos.
- **Definición de llaves primarias y foráneas** para asegurar la integridad referencial del modelo dimensional.

Este proceso permite establecer un esquema estrella que facilita consultas analíticas, generación de reportes y análisis de desempeño logístico en el sistema de mensajería.

---

## Integrantes del equipo

- Cristian David Cabrera  
- Andres Felipe Asprilla  
- Daniel Arias Castrillon
- Juan David Jaramillo  
- Brayan Esteven Narvaez 

---

## Requisitos

Antes de ejecutar este notebook, asegúrate de:

- Tener un archivo `.env` con las variables necesarias para conectarte a la base de datos.
- Haber creado previamente la base de datos de destino (bodega).
- Tener las bibliotecas `sqlalchemy`, `psycopg2-binary`, `python-dotenv` y `pandas` instaladas.

---


## Instalación de dependencias

Antes de ejecutar el proceso ETL, es necesario asegurarse de que las bibliotecas requeridas estén instaladas en el entorno de ejecución.

Las siguientes dependencias son fundamentales para este notebook:

- `python-dotenv`: Para cargar variables de entorno desde un archivo `.env`.
- `sqlalchemy`: Para gestionar las conexiones y operaciones con bases de datos usando un ORM ligero.
- `psycopg2-binary`: Driver para conectarse a bases de datos PostgreSQL desde Python.
- `pandas`: Librería para manipulación y análisis de datos tabulares.

Puedes instalar todas estas dependencias ejecutando la siguiente celda (descomentándola si estás en un entorno limpio):


In [1]:
#!pip install python-dotenv sqlalchemy psycopg2-binary pandas

## Instalación de dependencias

Para ejecutar este proceso ETL, es necesario instalar algunas bibliotecas de Python que permiten la conexión con bases de datos PostgreSQL, la gestión de variables de entorno y el manejo de datos en estructuras tipo DataFrame. Las siguientes librerías deben estar instaladas:

- `python-dotenv`: para cargar variables de entorno desde archivos `.env`.
- `sqlalchemy`: para gestionar conexiones a bases de datos de manera flexible.
- `psycopg2-binary`: controlador de PostgreSQL para Python.
- `pandas`: para la manipulación y análisis de datos.

La siguiente celda ejecuta el comando necesario para instalar estas dependencias (solo es necesario si aún no están instaladas en el entorno).


In [ ]:
import os
import pandas as pd
import psycopg2 
from psycopg2 import errors
from dotenv import load_dotenv
from sqlalchemy import create_engine

## Carga de variables de entorno y configuración de conexión

Para mantener segura y flexible la información sensible del proyecto (como credenciales y nombres de bases de datos), utilizamos un archivo `.env`. Con la ayuda de la librería `python-dotenv`, cargamos estas variables de entorno directamente en el entorno de ejecución.

En esta sección se realiza lo siguiente:

- Se cargan las variables de entorno que contienen el usuario, contraseña, host, puerto y los nombres de la base de datos de origen y de la bodega de datos.
- Se imprimen los nombres de ambas bases de datos como verificación.

Esto permite establecer conexiones a las bases de datos sin exponer directamente la información confidencial en el código fuente.


In [ ]:
# Cargar las variables de entorno desde .env
load_dotenv()

# Conexión general
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST")
port = os.getenv("DB_PORT")

# Base de datos origen
dbname = os.getenv("DB_NAME")

# Bodega de datos
dwname = os.getenv("DW_NAME")

print(f"{dbname} {dwname}")


Base_Datos_Proyecto Bodega_Datos_Proyecto


## Creación de motores de conexión (`engine`)

A continuación, se configuran los motores de conexión (`engine`) utilizando SQLAlchemy, una herramienta que permite interactuar con bases de datos de forma eficiente desde Python.

- `engine_db`: motor de conexión para acceder a la **base de datos de origen**.
- `engine_dw`: motor de conexión para interactuar con la **bodega de datos**.

Ambos motores utilizan las credenciales cargadas previamente desde las variables de entorno. Esta configuración es esencial para ejecutar consultas y transferir datos entre las bases de datos durante el proceso ETL.


In [4]:
# Engine para base origen
engine_db = create_engine(f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}")

# Engine para la bodega de datos
engine_dw = create_engine(f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{dwname}")


In [ ]:
query = "SELECT table_name FROM information_schema.tables WHERE table_schema = 'public' AND table_type = 'BASE TABLE';"

df = pd.read_sql(query, engine_dw)

df.head()

,table_name
0,dim_mensajero
1,dim_sede
2,dim_fecha
3,dim_hora
4,dim_cliente


## Construcción de la dimensión `cliente`

En esta sección se realiza el proceso de extracción, transformación y carga (ETL) para construir la dimensión `cliente` en la bodega de datos. A continuación se detallan los pasos seguidos:

1. **Extracción de datos**:  
   Se leen las tablas `cliente`, `ciudad` y `departamento` desde la base de datos original utilizando `pandas.read_sql()`.

2. **Transformación de nombres de columnas**:  
   Se renombran algunas columnas para estandarizar los nombres y facilitar los posteriores procesos de unión (join).

3. **Unión de tablas**:  
   Se realizan joins utilizando `pandas.merge()` para combinar la información del cliente con su ciudad y departamento correspondiente.

4. **Selección y ordenamiento**:  
   Se seleccionan únicamente las columnas relevantes para la dimensión y se ordenan los registros por `cliente_id`.

5. **Generación de llave primaria**:  
   Se crea una columna `key_dim_cliente` como identificador incremental único para la dimensión.

6. **Carga en la bodega de datos**:  
   La tabla resultante se guarda como `dim_cliente` en la bodega de datos, reemplazando la tabla si ya existía.

7. **Verificación**:  
   Se visualizan las primeras filas de la dimensión resultante para confirmar que el proceso se ha realizado correctamente.


In [6]:
# 1. Leer las tablas necesarias desde la base de datos original
df_cliente = pd.read_sql("SELECT * FROM cliente;", con=engine_db)
df_ciudad = pd.read_sql("SELECT * FROM ciudad;", con=engine_db)
df_departamento = pd.read_sql("SELECT * FROM departamento;", con=engine_db)

# 2. Renombrar columnas necesarias para estandarizar
df_ciudad = df_ciudad.rename(columns={
    'ciudad_id': 'id',
    'nombre': 'nombre_ciudad',
    'departamento_id': 'departamento_id'
})

df_departamento = df_departamento.rename(columns={
    'departamento_id': 'id',
    'nombre': 'nombre_departamento'
})

df_cliente = df_cliente.rename(columns={
    'nombre': 'nombre_cliente'
})

# 3. Realizar joins con pandas para construir la dimensión cliente
df_dim_cliente = df_cliente \
    .merge(df_ciudad, left_on='ciudad_id', right_on='id', suffixes=('', '_ciudad')) \
    .merge(df_departamento, left_on='departamento_id', right_on='id', suffixes=('', '_depto'))

# 4. Seleccionar columnas deseadas y ordenar
df_dim_cliente = df_dim_cliente[['cliente_id', 'nit_cliente', 'nombre_cliente']]
df_dim_cliente = df_dim_cliente.sort_values(by='cliente_id').reset_index(drop=True)

# 5. Crear la columna llave incremental
df_dim_cliente.insert(0, 'key_dim_cliente', range(len(df_dim_cliente)))

# 6. Guardar la tabla en la bodega de datos
df_dim_cliente.to_sql('dim_cliente', con=engine_dw, if_exists='replace', index=False)

# 7. Verificar
df_dim_cliente.head()



,key_dim_cliente,cliente_id,nit_cliente,nombre_cliente
0,0,1,25,Cliente 2
1,1,2,123,Cliente 1
2,2,3,312289-5,BANCO REGIONAL DE SANGRE BLOD-LIFE
3,3,4,306215-0,CRUZ AZUL-LIFE
4,4,5,300513-3,CLINICA CALI -JOVEN


## Definición de la clave primaria en la dimensión `cliente`

Una vez cargada la tabla `dim_cliente` en la bodega de datos, es fundamental garantizar su integridad y unicidad mediante la definición de una clave primaria.

En esta sección:

- Se establece una conexión directa a la bodega de datos utilizando `psycopg2`.
- Se crea un cursor para ejecutar comandos SQL.
- Se intenta añadir una **clave primaria** (`PRIMARY KEY`) sobre la columna `key_dim_cliente`.
- Se maneja la excepción `DuplicateObject` en caso de que la restricción ya exista.
- Finalmente, se cierra la conexión a la base de datos de forma segura.

Este paso asegura que la tabla `dim_cliente` pueda ser utilizada como dimensión en consultas de tipo OLAP, preservando la unicidad de cada registro.


In [7]:
# Conectar a la bodega de datos
conn = psycopg2.connect(
    dbname=dwname,
    user=user,
    password=password,
    host=host,
    port=port
)

# Crear cursor
cur = conn.cursor()

try:
    # Intentar añadir la primary key
    cur.execute("""
        ALTER TABLE dim_cliente
        ADD CONSTRAINT pk_dim_cliente PRIMARY KEY (key_dim_cliente);
    """)
    conn.commit()
    print("Primary key creada exitosamente.")

except errors.DuplicateObject:
    print("Ya existe una primary key en la tabla 'dim_cliente'. No se realizó ningún cambio.")

except Exception as e:
    print("Ocurrió un error:", e)

finally:
    # Cerrar cursor y conexión
    cur.close()
    conn.close()

Primary key creada exitosamente.


## Construcción de la dimensión `mensajero`

Esta sección del proceso ETL se enfoca en construir la dimensión `mensajero`, la cual contiene información clave sobre los mensajeros registrados en el sistema. A continuación, se describen los pasos realizados:

1. **Extracción de datos**:  
   Se recuperan las tablas `clientes_mensajeroaquitoy`, `auth_user`, `ciudad` y `departamento` desde la base de datos de origen.

2. **Normalización de nombres de columnas**:  
   Se renombran columnas en las tablas de ciudades y departamentos para facilitar los posteriores procesos de unión (`merge`).

3. **Transformación y combinación de datos**:  
   Utilizando `pandas.merge()`, se integran las tablas para enriquecer la información del mensajero con su usuario, ciudad y departamento de operación.

4. **Selección y renombramiento de columnas finales**:  
   Se seleccionan únicamente las columnas relevantes y se renombran para mayor claridad y coherencia con el diseño dimensional.

5. **Ordenamiento y generación de clave primaria**:  
   Se ordena la tabla por `mensajero_id` y se añade la columna `key_dim_mensajero`, que actúa como identificador único incremental.

6. **Carga en la bodega de datos**:  
   La tabla resultante se almacena como `dim_mensajero` en la bodega de datos.

7. **Verificación**:  
   Se visualizan las primeras filas de la dimensión generada para asegurar que los datos se han procesado correctamente.

Este proceso permite disponer de una dimensión limpia y estructurada que puede ser utilizada para análisis posteriores relacionados con la operación de los mensajeros.


In [8]:
# 1. Leer las tablas necesarias desde la base de datos origen
df_mensajero   = pd.read_sql("SELECT * FROM clientes_mensajeroaquitoy;", con=engine_db)
df_auth_user   = pd.read_sql("SELECT id, username FROM auth_user;", con=engine_db)
df_ciudad      = pd.read_sql("SELECT * FROM ciudad;", con=engine_db)
df_departamento = pd.read_sql("SELECT * FROM departamento;", con=engine_db)

# 2. Renombrar columnas para normalizar nombres
df_ciudad = df_ciudad.rename(columns={
    'ciudad_id': 'id',
    'nombre':    'nombre_ciudad',
    'departamento_id': 'departamento_id'
})

df_departamento = df_departamento.rename(columns={
    'departamento_id': 'id',
    'nombre':          'nombre_departamento'
})

# 3. Construir la dimensión mensajero con merges de pandas
df_dim_mensajero = (
    df_mensajero
    .merge(df_auth_user, left_on='user_id', right_on='id', suffixes=('', '_user'))
    .merge(df_ciudad,  how='left', left_on='ciudad_operacion_id', right_on='id', suffixes=('', '_ciudad'))
    .merge(df_departamento, how='left', left_on='departamento_id', right_on='id', suffixes=('', '_depto'))
)

# 4. Seleccionar y renombrar columnas finales
df_dim_mensajero = df_dim_mensajero[[ 
    'id',
    'username',
    'activo',
    'fecha_entrada',
    'nombre_ciudad',
    'nombre_departamento'
]].rename(columns={
    'id': 'mensajero_id',
    'username': 'nombre',
    'nombre_ciudad': 'ciudad_operacion',
    'nombre_departamento': 'departamento_operacion'
})

# 5. Ordenar por nombre y crear llave incremental
df_dim_mensajero = df_dim_mensajero.sort_values(by='mensajero_id').reset_index(drop=True)
df_dim_mensajero.insert(0, 'key_dim_mensajero', range(len(df_dim_mensajero)))

# 6. Cargar en la bodega de datos
df_dim_mensajero.to_sql('dim_mensajero', con=engine_dw, if_exists='replace', index=False)

# 7. Verificar
df_dim_mensajero.head()


,key_dim_mensajero,mensajero_id,nombre,activo,fecha_entrada,ciudad_operacion,departamento_operacion
0,0,1,mensajero1,True,None,ACOPI YUMBO,VALLE DEL CAUCA
1,1,2,mensajero2,True,None,NaN,NaN
2,2,3,Biil-Gates,True,2012-05-08,CALI,VALLE DEL CAUCA
3,3,4,Lionel_messi,False,2018-12-17,CALI,VALLE DEL CAUCA
4,4,5,James Rodriguez,True,2015-07-01,NaN,NaN


## Definición de la clave primaria en la dimensión `mensajero`

Una vez construida y cargada la dimensión `dim_mensajero` en la bodega de datos, se procede a definir la clave primaria sobre la columna `key_dim_mensajero`.

Este paso incluye:

- La conexión a la bodega de datos utilizando `psycopg2`.
- La ejecución de un comando SQL para añadir la restricción `PRIMARY KEY`.
- El manejo de excepciones, como el caso en que la restricción ya exista.
- El cierre adecuado de la conexión y el cursor.

Establecer esta clave primaria garantiza la integridad de los datos y permite que la dimensión `mensajero` pueda ser relacionada con otras tablas de hechos en futuros procesos de análisis.


In [9]:
# Conectar a la bodega de datos
conn = psycopg2.connect(
    dbname=dwname,
    user=user,
    password=password,
    host=host,
    port=port
)

# Crear cursor
cur = conn.cursor()

try:
    # Intentar añadir la primary key
    cur.execute("""
        ALTER TABLE dim_mensajero
        ADD CONSTRAINT pk_dim_mensajero PRIMARY KEY (key_dim_mensajero);
    """)
    conn.commit()
    print("Primary key creada exitosamente.")

except errors.DuplicateObject:
    print("Ya existe una primary key en la tabla 'dim_mensajero'. No se realizó ningún cambio.")

except Exception as e:
    print("Ocurrió un error:", e)

finally:
    # Cerrar cursor y conexión
    cur.close()
    conn.close()

Primary key creada exitosamente.


## Construcción de la dimensión `sede`

En esta parte del proceso ETL se construye la dimensión `sede`, que representa las distintas sedes de operación del sistema. El procedimiento incluye los siguientes pasos:

1. **Extracción de datos**:  
   Se consultan las tablas `sede`, `ciudad` y `departamento` desde la base de datos de origen.

2. **Normalización de columnas**:  
   Se renombran columnas en las tablas de ciudad, departamento y sede para mantener consistencia con el resto del modelo dimensional.

3. **Unión de datos**:  
   Se realizan uniones entre las tablas para obtener una vista enriquecida de cada sede, incluyendo su ciudad y departamento correspondiente.

4. **Selección y renombramiento de columnas**:  
   Se extraen únicamente las columnas relevantes para la dimensión y se renombran para mejorar la claridad semántica.

5. **Ordenamiento y creación de clave primaria**:  
   La tabla se ordena por `sede_id` y se agrega la columna `key_dim_sede`, que actúa como identificador único incremental.

6. **Carga en la bodega de datos**:  
   Se guarda la tabla resultante como `dim_sede` en la bodega de datos, sobrescribiendo su contenido si ya existía.

7. **Verificación**:  
   Se visualizan los primeros registros de la dimensión para comprobar que el resultado es coherente.

Esta dimensión será útil para analizar operaciones o servicios en función de la ubicación geográfica de cada sede.


In [10]:
# 1. Leer las tablas necesarias desde la base de datos origen
df_sede        = pd.read_sql("SELECT * FROM sede;", con=engine_db)
df_ciudad      = pd.read_sql("SELECT * FROM ciudad;", con=engine_db)
df_departamento = pd.read_sql("SELECT * FROM departamento;", con=engine_db)

# 2. Renombrar columnas para mantener consistencia
df_ciudad = df_ciudad.rename(columns={
    'ciudad_id': 'id',
    'nombre': 'nombre_ciudad',
    'departamento_id': 'departamento_id'
})

df_departamento = df_departamento.rename(columns={
    'departamento_id': 'id',
    'nombre': 'nombre_departamento'
})

df_sede = df_sede.rename(columns={
    'nombre': 'nombre_sede'
})

# 3. Construir la dimensión sede haciendo los joins necesarios
df_dim_sede = (
    df_sede
    .merge(df_ciudad, left_on='ciudad_id', right_on='id', suffixes=('', '_ciudad'))
    .merge(df_departamento, left_on='departamento_id', right_on='id', suffixes=('', '_depto'))
)

# 4. Seleccionar y renombrar las columnas finales
df_dim_sede = df_dim_sede[[ 
    'sede_id',
    'nombre_sede',
    'direccion',
    'nombre_ciudad',
    'nombre_departamento'
]].rename(columns={
    'nombre_ciudad': 'ciudad_sede',
    'nombre_departamento': 'departamento_sede'
})

# 5. Ordenar por nombre_sede y agregar la columna key_dim_sede
df_dim_sede = df_dim_sede.sort_values(by='sede_id').reset_index(drop=True)
df_dim_sede.insert(0, 'key_dim_sede', range(len(df_dim_sede)))


# 6. Cargar en la bodega de datos
df_dim_sede.to_sql('dim_sede', con=engine_dw, if_exists='replace', index=False)

# 7. Verificar visualizando las primeras filas
df_dim_sede.head()


,key_dim_sede,sede_id,nombre_sede,direccion,ciudad_sede,departamento_sede
0,0,1,Sede principal - Cliente1,Los angeles distrito Latino,CALI,VALLE DEL CAUCA
1,1,2,sede aux - cliente 1,Los angeles distrito Latino,CALI,VALLE DEL CAUCA
2,2,3,TORRES DE MARACAIBO,Los angeles distrito Latino,CALI,VALLE DEL CAUCA
3,3,4,INGENIO,Los angeles distrito Latino,CALI,VALLE DEL CAUCA
4,4,5,VASQUEZ COBO,Los angeles distrito Latino,CALI,VALLE DEL CAUCA


## Definición de la clave primaria en la dimensión `sede`

Después de cargar la dimensión `dim_sede` en la bodega de datos, es importante establecer una clave primaria que garantice la unicidad de los registros.

Este paso realiza lo siguiente:

- Se conecta a la bodega de datos mediante `psycopg2`.
- Se ejecuta una instrucción `ALTER TABLE` para añadir la restricción `PRIMARY KEY` sobre la columna `key_dim_sede`.
- Se maneja el error `DuplicateObject` en caso de que la clave primaria ya exista.
- Finalmente, se cierra de forma segura tanto el cursor como la conexión a la base de datos.

Este procedimiento asegura la integridad referencial de la dimensión `sede`, habilitándola para participar en relaciones con tablas de hechos dentro del esquema estrella de la bodega de datos.


In [11]:
# Conectar a la bodega de datos
conn = psycopg2.connect(
    dbname=dwname,
    user=user,
    password=password,
    host=host,
    port=port
)

# Crear cursor
cur = conn.cursor()

try:
    # Intentar añadir la primary key
    cur.execute("""
        ALTER TABLE dim_sede
        ADD CONSTRAINT pk_dim_sede PRIMARY KEY (key_dim_sede);
    """)
    conn.commit()
    print("Primary key creada exitosamente.")

except errors.DuplicateObject:
    print("Ya existe una primary key en la tabla 'dim_sede'. No se realizó ningún cambio.")

except Exception as e:
    print("Ocurrió un error:", e)

finally:
    # Cerrar cursor y conexión
    cur.close()
    conn.close()


Primary key creada exitosamente.


## Construcción de la dimensión `fecha`

La dimensión `fecha` es una de las más importantes en cualquier modelo de bodega de datos, ya que permite realizar análisis temporales y agrupar medidas por día, mes, trimestre o año.

A continuación se describen los pasos realizados para construirla:

1. **Generación del rango de fechas**:  
   Se crea un rango diario que abarca desde el 1 de enero de 2023 hasta el 31 de diciembre de 2024, utilizando `pandas.date_range()`.

2. **Creación del DataFrame base**:  
   Se genera un DataFrame a partir del rango de fechas.

3. **Extracción de componentes temporales**:  
   Se añaden columnas con información útil para el análisis:
   - `key_dim_fecha`: identificador único incremental.
   - `dia`: día del mes.
   - `mes`: número del mes.
   - `nombre_mes`: nombre del mes en español.
   - `nombre_dia`: nombre del día de la semana en español.

4. **Reordenamiento de columnas**:  
   Se estructura el DataFrame con el orden adecuado para su uso como dimensión temporal.

5. **Carga en la bodega de datos**:  
   La tabla se guarda como `dim_fecha` en la bodega de datos, sobrescribiendo su contenido si ya existía.

6. **Verificación**:  
   Se muestran los primeros 10 registros de la dimensión para asegurar su correcta construcción.

Esta dimensión permite realizar análisis como comparaciones entre meses, días de la semana o patrones estacionales en las tablas de hechos.


In [ ]:
# 1. Crear un rango de fechas para 2023 y 2024
fechas = pd.date_range(start='2023-01-01', end='2024-12-31', freq='D')

# 2. Crear dataframe con la columna fecha
df_dim_fecha = pd.DataFrame({'fecha': fechas})

# 3. Extraer componentes: id, dia, mes, nombre_mes, nombre_dia
df_dim_fecha['key_dim_fecha'] = df_dim_fecha.index  # O algún id secuencial
df_dim_fecha['dia'] = df_dim_fecha['fecha'].dt.day
df_dim_fecha['mes'] = df_dim_fecha['fecha'].dt.month
df_dim_fecha['nombre_mes'] = df_dim_fecha['fecha'].dt.month_name(locale='es_ES')  # nombre del mes en español
df_dim_fecha['nombre_dia'] = df_dim_fecha['fecha'].dt.day_name(locale='es_ES')    # nombre del día en español

# 4. Reordenar columnas 
df_dim_fecha = df_dim_fecha[['key_dim_fecha', 'fecha', 'dia', 'mes', 'nombre_mes', 'nombre_dia']]

# 5. Guardar la dimensión en la bodega de datos
df_dim_fecha.to_sql('dim_fecha', con=engine_dw, if_exists='replace', index=False)

# 6. Mostrar resultado
print(df_dim_fecha.head(10))



   key_dim_fecha      fecha  dia  mes nombre_mes nombre_dia
0              0 2023-01-01    1    1      Enero    Domingo
1              1 2023-01-02    2    1      Enero      Lunes
2              2 2023-01-03    3    1      Enero     Martes
3              3 2023-01-04    4    1      Enero  Miércoles
4              4 2023-01-05    5    1      Enero     Jueves
5              5 2023-01-06    6    1      Enero    Viernes
6              6 2023-01-07    7    1      Enero     Sábado
7              7 2023-01-08    8    1      Enero    Domingo
8              8 2023-01-09    9    1      Enero      Lunes
9              9 2023-01-10   10    1      Enero     Martes


## Definición de la clave primaria en la dimensión `fecha`

Una vez construida y cargada la dimensión `dim_fecha`, se procede a establecer la clave primaria sobre la columna `key_dim_fecha`, que actúa como identificador único de cada fecha.

Esta parte del proceso incluye:

- La conexión a la bodega de datos mediante `psycopg2`.
- La ejecución de un comando `ALTER TABLE` para definir la restricción `PRIMARY KEY`.
- El manejo de excepciones, específicamente si la clave ya ha sido definida previamente.
- El cierre ordenado del cursor y la conexión.

Definir esta clave primaria es fundamental para garantizar la unicidad de los registros y permitir relaciones consistentes con tablas de hechos que dependan del tiempo.


In [13]:
# Conectar a la bodega de datos
conn = psycopg2.connect(
    dbname=dwname,
    user=user,
    password=password,
    host=host,
    port=port
)

# Crear cursor
cur = conn.cursor()

try:
    # Intentar añadir la primary key
    cur.execute("""
        ALTER TABLE dim_fecha
        ADD CONSTRAINT pk_dim_fecha PRIMARY KEY (key_dim_fecha);
    """)
    conn.commit()
    print("Primary key creada exitosamente.")

except errors.DuplicateObject:
    print("Ya existe una primary key en la tabla 'dim_fecha'. No se realizó ningún cambio.")

except Exception as e:
    print("Ocurrió un error:", e)

finally:
    # Cerrar cursor y conexión
    cur.close()
    conn.close()

Primary key creada exitosamente.


## Construcción de la dimensión `hora`

La dimensión `hora` permite realizar análisis de datos a nivel temporal detallado dentro del día (por ejemplo, distribución de eventos por hora, minuto o segundo).

En esta sección se realiza lo siguiente:

1. **Generación del rango de segundos en un día**:  
   Se consideran los 86.400 segundos que componen un día (24 horas x 60 minutos x 60 segundos).

2. **Conversión a formato de tiempo**:  
   Cada segundo se transforma en una duración (`Timedelta`) y se extraen las componentes de hora, minuto y segundo.

3. **Formateo de la hora**:  
   Se crea una columna con el formato `HH:MM:SS` para facilitar la legibilidad y posibles visualizaciones.

4. **Asignación de clave primaria**:  
   Se agrega una columna `key_dim_hora` como identificador incremental único para cada registro.

5. **Reorganización de columnas**:  
   Se estructura el DataFrame en un orden lógico: clave, hora formateada y componentes individuales.

6. **Carga en la bodega de datos**:  
   La dimensión resultante se guarda en la tabla `dim_hora` de la bodega de datos, sobrescribiendo su contenido si ya existía.

7. **Verificación**:  
   Se muestran las primeras filas del DataFrame para confirmar que la construcción es correcta.

Esta dimensión permite realizar análisis temporales precisos a nivel de segundo, útil en aplicaciones donde se requiere granularidad fina del tiempo.


In [64]:
# Crear un rango de tiempo para todas las horas del día con segundos
# Total segundos en un día = 24*60*60 = 86400
total_seconds = 24 * 60 * 60

# Crear un DataFrame con todas las horas, minutos y segundos del día
df_hora = pd.DataFrame({
    'segundos_del_dia': range(total_seconds)
})

# Convertir segundos del día a tiempo
df_hora['hora_completa'] = pd.to_timedelta(df_hora['segundos_del_dia'], unit='s')

# Extraer hora, minuto y segundo
df_hora['hora'] = df_hora['hora_completa'].dt.components['hours']
df_hora['minuto'] = df_hora['hora_completa'].dt.components['minutes']
df_hora['segundo'] = df_hora['hora_completa'].dt.components['seconds']

# Crear columna con hora en formato HH:MM:SS
df_hora['hora_formateada'] = df_hora['hora_completa'].astype(str).str.slice(7, 15)  # tipo 'hh:mm:ss'
df_hora['hora_formateada'] = pd.to_datetime(df_hora['hora_formateada'], format='%H:%M:%S').dt.time

# Asignar un id único (opcional)
df_hora['key_dim_hora'] = df_hora.index

# Reordenar columnas para mejor legibilidad
df_hora = df_hora[['key_dim_hora', 'hora_formateada', 'hora', 'minuto', 'segundo']]

# Guardar en la bodega de datos
df_hora.to_sql('dim_hora', con=engine_dw, if_exists='replace', index=False)

# Mostrar primeras filas
print(df_hora.head())



   key_dim_hora hora_formateada  hora  minuto  segundo
0             0        00:00:00     0       0        0
1             1        00:00:01     0       0        1
2             2        00:00:02     0       0        2
3             3        00:00:03     0       0        3
4             4        00:00:04     0       0        4


## Definición de la clave primaria en la dimensión `hora`

Después de construir y cargar la tabla `dim_hora` en la bodega de datos, se define una clave primaria para garantizar la unicidad de cada registro.

Este procedimiento incluye:

- La conexión a la bodega de datos usando `psycopg2`.
- La ejecución de una sentencia SQL `ALTER TABLE` para definir la restricción `PRIMARY KEY` sobre la columna `key_dim_hora`.
- El manejo del error `DuplicateObject` en caso de que la clave ya exista.
- El cierre seguro del cursor y la conexión.

Establecer esta clave primaria es esencial para asegurar la integridad de los datos y habilitar relaciones consistentes con tablas de hechos que requieran un nivel de detalle temporal por segundo.


In [72]:
# Conectar a la bodega de datos
conn = psycopg2.connect(
    dbname=dwname,
    user=user,
    password=password,
    host=host,
    port=port
)

# Crear cursor
cur = conn.cursor()

try:
    # Intentar añadir la primary key
    cur.execute("""
        ALTER TABLE dim_hora
        ADD CONSTRAINT pk_dim_hora PRIMARY KEY (key_dim_hora);
    """)
    conn.commit()
    print("Primary key creada exitosamente.")

except errors.DuplicateObject:
    print("Ya existe una primary key en la tabla 'dim_hora'. No se realizó ningún cambio.")

except Exception as e:
    print("Ocurrió un error:", e)

finally:
    # Cerrar cursor y conexión
    cur.close()
    conn.close()

Primary key creada exitosamente.


## Construcción de la tabla de hechos `hecho_mensajeria_servicio`

En esta sección se construye la tabla de hechos principal para el modelo de bodega de datos, la cual contiene información detallada sobre el ciclo de vida de cada servicio de mensajería, incluyendo relaciones con múltiples dimensiones y momentos clave del proceso.

### Pasos realizados:

1. **Extracción de datos**:  
   Se leen las tablas relacionadas con servicios, estados, usuarios, tipos de servicio, y se renombran columnas para facilitar su integración.

2. **Integración de información**:  
   Se realiza una serie de `merge()` para enriquecer los datos de servicios con la información del cliente, tipo de servicio, estado del servicio y demás atributos necesarios.

3. **Transformación a estructura de hechos**:  
   Se utilizan tablas dinámicas (`pivot_table`) para convertir los estados del servicio en columnas que representen las fechas y horas asociadas a cada evento (por ejemplo: iniciado, asignado, entregado, etc.).

4. **Unión con dimensiones**:  
   Se agregan las claves foráneas desde las dimensiones `cliente`, `mensajero`, `sede`, `fecha` y `hora`, mediante procesos de unión personalizados para vincular correctamente los datos temporales.

5. **Limpieza y organización**:  
   Se eliminan columnas redundantes, se reordenan las columnas para mejorar la legibilidad, y se convierte la información temporal en claves enteras (`key_dim_fecha`, `key_dim_hora`).

6. **Generación de clave primaria y medida**:  
   Se crea una columna `key_hecho_mensajeria_servicio` como identificador único de la fila, y se añade la medida `cantidad_servicios`, que representa una ocurrencia individual por registro.

7. **Carga en la bodega de datos**:  
   Finalmente, se guarda la tabla `hecho_mensajeria_servicio` en la bodega de datos, reemplazando su contenido si ya existía.

Esta tabla de hechos constituye el núcleo del modelo analítico, permitiendo realizar consultas sobre tiempos de entrega, frecuencia de estados, desempeño por sede, y mucho más, gracias a sus múltiples relaciones con las dimensiones construidas previamente.


In [118]:
# 1. Leer las tablas
df_estado_servicio = pd.read_sql("SELECT * FROM mensajeria_estadosservicio;", con=engine_db)
df_servicio = pd.read_sql("SELECT * FROM mensajeria_servicio;", con=engine_db)
df_estado = pd.read_sql("SELECT * FROM mensajeria_estado;", con=engine_db)
df_usuario = pd.read_sql("SELECT * FROM clientes_usuarioaquitoy;", con=engine_db)
df_tipo_servicio = pd.read_sql("SELECT * FROM mensajeria_tiposervicio;", con=engine_db)

df_tipo_servicio.rename(columns={'nombre': 'tipo_servicio'}, inplace=True)


# 2. Unir para tener toda la información
df_servicio = df_servicio.merge(df_usuario, left_on='usuario_id', right_on='id', suffixes=('', '_usuario'))
df_servicio = df_servicio.merge(df_tipo_servicio, left_on='tipo_servicio_id', right_on='id', suffixes=('', '_tipo_servicio'))
df_estado_servicio = df_estado_servicio.merge(df_estado, left_on='estado_id', right_on='id', suffixes=('', '_estado'))
df_estado_servicio = df_estado_servicio.merge(df_servicio, left_on='servicio_id', right_on='id', suffixes=('', '_servicio'))

# 3. Pivotear el DataFrame para tener una columna por estado
df_pivot_fecha = df_estado_servicio.pivot_table(index='servicio_id', columns='nombre', values='fecha', aggfunc='max')
df_pivot_hora = df_estado_servicio.pivot_table(index='servicio_id', columns='nombre', values='hora', aggfunc='max')

# 4. Renombrar columnas
df_pivot_fecha.columns = [f"{col.lower()}_fecha".replace(" ", "_").lower() for col in df_pivot_fecha.columns]
df_pivot_hora.columns = [f"{col.lower()}_hora".replace(" ", "_").lower() for col in df_pivot_hora.columns]

# 5. Combinar las columnas de fecha y hora
df_hecho_mensajeria_servicio = pd.concat([df_pivot_fecha, df_pivot_hora], axis=1).reset_index()

# 6. Agregar llaves foráneas desde dimensiones
df_dim = df_estado_servicio[['servicio_id', 'sede_id', 'cliente_id', 'mensajero_id', 'tipo_servicio']].drop_duplicates()
df_hecho_mensajeria_servicio = df_hecho_mensajeria_servicio.merge(df_dim, on='servicio_id', how='left')

df_hecho_mensajeria_servicio = df_hecho_mensajeria_servicio \
    .merge(df_dim_sede, on='sede_id') \
    .merge(df_dim_cliente, on='cliente_id') \
    .merge(df_dim_mensajero, on='mensajero_id')

# 7. Seleccionar columnas finales
df_hecho_mensajeria_servicio = df_hecho_mensajeria_servicio[[
    'servicio_id',
    'key_dim_sede',
    'key_dim_cliente',
    'key_dim_mensajero',
    'tipo_servicio',
    *[col for col in df_hecho_mensajeria_servicio.columns if '_fecha' in col or '_hora' in col]
]]

# 8. Agregar llaves foraneas a las fechas 

primeras_columnas = [col for col in df_hecho_mensajeria_servicio.columns if "_fecha" in col]

for col in primeras_columnas:
    df_hecho_mensajeria_servicio[col] = pd.to_datetime(df_hecho_mensajeria_servicio[col])

def reemplazar_fecha_por_llave(df_hechos, df_dim_fecha, nombre_columna):
    df_temp = df_hechos[[nombre_columna]].drop_duplicates()
    df_temp = df_temp.merge(df_dim_fecha, left_on=nombre_columna, right_on='fecha', how='left')
    df_temp = df_temp[[nombre_columna, 'key_dim_fecha']]
    df_temp = df_temp.rename(columns={'key_dim_fecha': f'key_{nombre_columna}'})
    return df_hechos.merge(df_temp, on=nombre_columna, how='left')

for col in primeras_columnas:
    df_hecho_mensajeria_servicio = reemplazar_fecha_por_llave(df_hecho_mensajeria_servicio, df_dim_fecha, col)

df_hecho_mensajeria_servicio = df_hecho_mensajeria_servicio.drop(columns=primeras_columnas)

# 9. Agregar llaves foraneas a las horas

primeras_columnas = [col for col in df_hecho_mensajeria_servicio.columns if "_hora" in col]

def reemplazar_hora_por_llave(df_hechos, df_hora, nombre_columna):
    df_temp = df_hechos[[nombre_columna]].drop_duplicates()
    df_temp = df_temp.merge(df_hora, left_on=nombre_columna, right_on='hora_formateada', how='left')
    df_temp = df_temp[[nombre_columna, 'key_dim_hora']]
    df_temp = df_temp.rename(columns={'key_dim_hora': f'key_{nombre_columna}'})
    return df_hechos.merge(df_temp, on=nombre_columna, how='left')

for col in primeras_columnas:
    df_hecho_mensajeria_servicio = reemplazar_hora_por_llave(df_hecho_mensajeria_servicio, df_hora, col)

df_hecho_mensajeria_servicio = df_hecho_mensajeria_servicio.drop(columns=primeras_columnas)

# 10. Organizamos las columnas para mejor lectura

df_hecho_mensajeria_servicio = df_hecho_mensajeria_servicio[[
    'servicio_id',
    'key_dim_cliente',
    'key_dim_mensajero',
    'key_dim_sede',
    'key_iniciado_fecha',
    'key_iniciado_hora',
    'key_con_mensajero_asignado_fecha',
    'key_con_mensajero_asignado_hora',
    'key_con_novedad_fecha',
    'key_con_novedad_hora',
    'key_recogido_por_mensajero_fecha',
    'key_recogido_por_mensajero_hora',
    'key_entregado_en_destino_fecha',
    'key_entregado_en_destino_hora', 
    'key_terminado_completo_fecha',
    'key_terminado_completo_hora',
    'tipo_servicio'
]]

# Agregar columna de llave primaria incremental
df_hecho_mensajeria_servicio.insert(0, 'key_hecho_mensajeria_servicio', range(len(df_hecho_mensajeria_servicio)))

# Convierte todas las claves de fecha y hora a enteros
for col in df_hecho_mensajeria_servicio.columns:
    if 'key_' in col and ('_fecha' in col or '_hora' in col):
        df_hecho_mensajeria_servicio[col] = df_hecho_mensajeria_servicio[col].astype('Int64')  # o int si no hay nulos

df_hecho_mensajeria_servicio['cantidad_servicios'] = 1

# Cargar en la bodega
df_hecho_mensajeria_servicio.to_sql('hecho_mensajeria_servicio', con=engine_dw, if_exists='replace', index=False)


703

## Definición de la clave primaria en la tabla de hechos `hecho_mensajeria_servicio`

Como paso final en la construcción de la tabla de hechos `hecho_mensajeria_servicio`, se define una clave primaria sobre la columna `key_hecho_mensajeria_servicio`.

Este procedimiento tiene como objetivo asegurar la integridad del modelo y permitir una identificación única de cada fila en la tabla de hechos.

El proceso incluye:

- La conexión a la bodega de datos mediante `psycopg2`.
- La ejecución de un comando SQL `ALTER TABLE` para establecer la restricción `PRIMARY KEY`.
- El manejo de excepciones, incluyendo el caso en que la clave primaria ya haya sido creada anteriormente.
- El cierre adecuado del cursor y de la conexión a la base de datos.

Establecer esta clave es fundamental para mantener la consistencia del modelo dimensional y para permitir relaciones confiables con las dimensiones correspondientes.


In [119]:
# Conectar a la bodega de datos
conn = psycopg2.connect(
    dbname=dwname,
    user=user,
    password=password,
    host=host,
    port=port
)

# Crear cursor
cur = conn.cursor()

try:
    # Intentar añadir la primary key
    cur.execute("""
        ALTER TABLE hecho_mensajeria_servicio
        ADD CONSTRAINT pk_hecho_mensajeria_servicio PRIMARY KEY (key_hecho_mensajeria_servicio);
    """)
    conn.commit()
    print("Primary key creada exitosamente.")

except errors.DuplicateObject:
    print("Ya existe una primary key en la tabla 'key_hecho_mensajeria_servicio'. No se realizó ningún cambio.")

except Exception as e:
    print("Ocurrió un error:", e)

finally:
    # Cerrar cursor y conexión
    cur.close()
    conn.close()

Primary key creada exitosamente.


## Definición de claves foráneas en la tabla de hechos `hecho_mensajeria_servicio`

Una vez construida la tabla de hechos, es fundamental establecer las **relaciones entre hechos y dimensiones** mediante claves foráneas. Este paso garantiza la integridad referencial del modelo de datos y permite realizar análisis cruzando información de diferentes dimensiones.

### Acciones realizadas:

1. **Conexión a la bodega de datos**:  
   Se establece una conexión mediante `psycopg2` para ejecutar sentencias SQL.

2. **Definición de claves foráneas a dimensiones principales**:  
   - `key_dim_sede` → `dim_sede(key_dim_sede)`
   - `key_dim_cliente` → `dim_cliente(key_dim_cliente)`
   - `key_dim_mensajero` → `dim_mensajero(key_dim_mensajero)`

3. **Definición de claves foráneas a la dimensión `fecha`**:  
   Se agregan restricciones de integridad para las columnas de fecha correspondientes a los distintos estados del servicio:
   - `iniciado`
   - `con_mensajero_asignado`
   - `con_novedad`
   - `recogido_por_mensajero`
   - `entregado_en_destino`
   - `terminado_completo`

4. **Definición de claves foráneas a la dimensión `hora`**:  
   Del mismo modo, se vinculan las horas de cada estado con la dimensión de tiempo a nivel de segundos.

5. **Manejo de errores**:  
   En caso de que alguna clave foránea ya exista, el código captura la excepción `DuplicateObject`.

6. **Cierre de la conexión**:  
   Se realiza el `commit()` de los cambios y se cierran el cursor y la conexión de forma segura.

Este paso concluye la modelación física del esquema estrella, permitiendo integridad entre hechos y dimensiones, así como la ejecución de consultas analíticas complejas de manera eficiente.


In [120]:
import psycopg2

conn = psycopg2.connect(
    dbname=dwname,
    user=user,
    password=password,
    host=host,
    port=port
)
cur = conn.cursor()

try:
    # FK a dimensiones: sede, cliente, mensajero
    cur.execute("""
        ALTER TABLE hecho_mensajeria_servicio
        ADD CONSTRAINT fk_sede FOREIGN KEY (key_dim_sede) REFERENCES dim_sede(key_dim_sede),
        ADD CONSTRAINT fk_cliente FOREIGN KEY (key_dim_cliente) REFERENCES dim_cliente(key_dim_cliente),
        ADD CONSTRAINT fk_mensajero FOREIGN KEY (key_dim_mensajero) REFERENCES dim_mensajero(key_dim_mensajero);
    """)

    # FK a dimensión fecha
    for estado in ['iniciado', 'con_mensajero_asignado', 'con_novedad',
                   'recogido_por_mensajero', 'entregado_en_destino', 'terminado_completo']:
        cur.execute(f"""
            ALTER TABLE hecho_mensajeria_servicio
            ADD CONSTRAINT fk_{estado}_fecha FOREIGN KEY (key_{estado}_fecha) REFERENCES dim_fecha(key_dim_fecha);
        """)
    
    # FK a dimensión hora
    for estado in ['iniciado', 'con_mensajero_asignado', 'con_novedad',
                   'recogido_por_mensajero', 'entregado_en_destino', 'terminado_completo']:
        cur.execute(f"""
            ALTER TABLE hecho_mensajeria_servicio
            ADD CONSTRAINT fk_{estado}_hora FOREIGN KEY (key_{estado}_hora) REFERENCES dim_hora(key_dim_hora);
        """)
    
except psycopg2.errors.DuplicateObject as e:
    print("Una o más llaves foráneas ya existen:", e)
finally:
    conn.commit()
    cur.close()
    conn.close()


## Construcción de la tabla de hechos `hecho_novedad`

La tabla de hechos `hecho_novedad` captura eventos relacionados con las novedades que ocurren durante la prestación de servicios de mensajería. Cada fila representa una novedad específica asociada a un servicio, e incluye detalles temporales, categorización y descripción del evento.

### Pasos realizados:

1. **Extracción de datos**:  
   Se leen las tablas relacionadas con novedades, servicios, tipos de novedad y usuarios.

2. **Integración de información**:  
   Se realiza uniones (`merge`) para enriquecer cada evento de novedad con los atributos del servicio, el tipo de novedad y la relación con otras entidades como el cliente, mensajero y sede.

3. **Separación de componentes temporales**:  
   - Se separa la fecha (`novedad_fecha`) truncando a `00:00:00`.
   - Se extrae la hora (`novedad_hora`) truncando a segundos, evitando fracciones de milisegundos que podrían complicar los joins.

4. **Agregación de claves foráneas**:
   - Se agregan claves de dimensión (`key_dim_sede`, `key_dim_cliente`, `key_dim_mensajero`).
   - Se reemplazan las columnas de fecha y hora con claves a las dimensiones `dim_fecha` y `dim_hora`, respectivamente.

5. **Selección y ordenamiento de columnas**:  
   Se seleccionan solo los campos necesarios para el análisis, reordenándolos para facilitar su lectura.

6. **Clave primaria y medida**:  
   Se añade `key_hecho_novedad` como identificador único de cada evento. Además, se agrega la medida `cantidad_novedades`, con valor 1 por defecto, para facilitar los análisis agregados (por ejemplo, contar el número de novedades por mensajero o por sede).

7. **Carga en la bodega de datos**:  
   La tabla final se guarda en la bodega de datos bajo el nombre `hecho_novedad`, sobrescribiendo cualquier contenido anterior.

Con esta tabla de hechos se habilita el análisis detallado de las novedades registradas durante los servicios, lo que puede aportar insights valiosos sobre problemas operativos, desempeño del personal y mejoras en la logística.


In [111]:
# 1. Leer las tablas
df_estado_novedad = pd.read_sql("SELECT * FROM mensajeria_novedadesservicio;", con=engine_db)
df_servicio = pd.read_sql("SELECT * FROM mensajeria_servicio;", con=engine_db)
df_usuario = pd.read_sql("SELECT * FROM clientes_usuarioaquitoy;", con=engine_db)
df_tipo_estado = pd.read_sql("SELECT * FROM mensajeria_tiponovedad;", con=engine_db)

df_tipo_estado.rename(columns={'nombre': 'tipo_novedad'}, inplace=True)

# 2. Unir para tener toda la información
df_servicio = df_servicio.merge(df_usuario, left_on='usuario_id', right_on='id', suffixes=('', '_usuario'))
df_estado_novedad = df_estado_novedad.merge(df_servicio, left_on='servicio_id', right_on='id', suffixes=('', '_servicio'))
df_estado_novedad = df_estado_novedad.merge(df_tipo_estado, left_on='tipo_novedad_id', right_on='id', suffixes=('', '_tipo_novedad'))

# Separar la fecha con hora 00:00:00
df_estado_novedad['novedad_fecha'] = pd.to_datetime(df_estado_novedad['fecha_novedad'].dt.date)  # Esto da un timestamp con hora 00:00:00

# Truncar a segundos antes de extraer la hora
df_estado_novedad['fecha_novedad_sin_frac'] = df_estado_novedad['fecha_novedad'].dt.floor('s')

# Extraer hora como datetime.time sin fracción de segundos
df_estado_novedad['novedad_hora'] = df_estado_novedad['fecha_novedad_sin_frac'].dt.time


# 5. Combinar las columnas de fecha y hora
df_hecho_novedad = df_estado_novedad



df_hecho_novedad = df_hecho_novedad \
    .merge(df_dim_sede, on='sede_id') \
    .merge(df_dim_cliente, on='cliente_id') \
    .merge(df_dim_mensajero, on='mensajero_id')

# 7. Seleccionar columnas finales
df_hecho_novedad = df_hecho_novedad[[
    'servicio_id',
    'key_dim_sede',
    'key_dim_cliente',
    'key_dim_mensajero',
    'novedad_fecha',
    'novedad_hora',
    'tipo_novedad',
    'descripcion'
]]


# 8. Agregar llaves foraneas a las fechas 

primeras_columnas = [col for col in df_hecho_novedad.columns if "_fecha" in col]

for col in primeras_columnas:
    df_hecho_novedad[col] = pd.to_datetime(df_hecho_novedad[col])

def reemplazar_fecha_por_llave(df_hechos, df_dim_fecha, nombre_columna):
    df_temp = df_hechos[[nombre_columna]].drop_duplicates()
    df_temp = df_temp.merge(df_dim_fecha, left_on=nombre_columna, right_on='fecha', how='left')
    df_temp = df_temp[[nombre_columna, 'key_dim_fecha']]
    df_temp = df_temp.rename(columns={'key_dim_fecha': f'key_{nombre_columna}'})
    return df_hechos.merge(df_temp, on=nombre_columna, how='left')

for col in primeras_columnas:
    df_hecho_novedad = reemplazar_fecha_por_llave(df_hecho_novedad, df_dim_fecha, col)

df_hecho_novedad = df_hecho_novedad.drop(columns=primeras_columnas)

# 9. Agregar llaves foraneas a las horas

primeras_columnas = [col for col in df_hecho_novedad.columns if "_hora" in col]

def reemplazar_hora_por_llave(df_hechos, df_hora, nombre_columna):
    df_temp = df_hechos[[nombre_columna]].drop_duplicates()
    df_temp = df_temp.merge(df_hora, left_on=nombre_columna, right_on='hora_formateada', how='left')
    df_temp = df_temp[[nombre_columna, 'key_dim_hora']]
    df_temp = df_temp.rename(columns={'key_dim_hora': f'key_{nombre_columna}'})
    return df_hechos.merge(df_temp, on=nombre_columna, how='left')

for col in primeras_columnas:
    df_hecho_novedad = reemplazar_hora_por_llave(df_hecho_novedad, df_hora, col)

df_hecho_novedad = df_hecho_novedad.drop(columns=primeras_columnas)

# Reorganizar columnas para mejor lectura
df_hecho_novedad = df_hecho_novedad[[
    'servicio_id',
    'key_dim_sede',
    'key_dim_cliente',
    'key_dim_mensajero',
    'key_novedad_fecha',
    'key_novedad_hora',
    'tipo_novedad',
    'descripcion'
]]

# Agregar columna de llave primaria incremental
df_hecho_novedad.insert(0, 'key_hecho_novedad', range(len(df_hecho_novedad)))

# Convierte todas las claves de fecha y hora a enteros
for col in df_hecho_novedad.columns:
    if 'key_' in col and ('_fecha' in col or '_hora' in col):
        df_hecho_novedad[col] = df_hecho_novedad[col].astype('Int64')  # o int si no hay nulos

df_hecho_novedad['cantidad_novedades'] = 1

# Cargar en la bodega
df_hecho_novedad.to_sql('hecho_novedad', con=engine_dw, if_exists='replace', index=False)




208

## Definición de la clave primaria en la tabla de hechos `hecho_novedad`

Luego de construir y poblar la tabla `hecho_novedad`, se define una clave primaria sobre la columna `key_hecho_novedad` con el fin de:

- Asegurar la unicidad de cada registro en la tabla de hechos.
- Facilitar las operaciones de join con otras tablas.
- Garantizar la integridad estructural del modelo dimensional.

### Proceso realizado:

- Se establece la conexión a la bodega de datos con `psycopg2`.
- Se ejecuta un comando SQL `ALTER TABLE` para añadir la restricción `PRIMARY KEY` sobre la columna `key_hecho_novedad`.
- Se maneja la excepción `DuplicateObject` en caso de que la restricción ya exista.
- Finalmente, se cierra correctamente la conexión y el cursor.

Este paso garantiza que la tabla de hechos sea robusta y confiable dentro del esquema estrella de la bodega de datos.


In [112]:
# Conectar a la bodega de datos
conn = psycopg2.connect(
    dbname=dwname,
    user=user,
    password=password,
    host=host,
    port=port
)

# Crear cursor
cur = conn.cursor()

try:
    # Intentar añadir la primary key
    cur.execute("""
        ALTER TABLE hecho_novedad
        ADD CONSTRAINT pk_hecho_novedad PRIMARY KEY (key_hecho_novedad);
    """)
    conn.commit()
    print("Primary key creada exitosamente.")

except errors.DuplicateObject:
    print("Ya existe una primary key en la tabla 'key_hecho_novedad'. No se realizó ningún cambio.")

except Exception as e:
    print("Ocurrió un error:", e)

finally:
    # Cerrar cursor y conexión
    cur.close()
    conn.close()

Primary key creada exitosamente.


## Definición de claves foráneas en la tabla de hechos `hecho_novedad`

Después de establecer la clave primaria, el siguiente paso consiste en definir las claves foráneas que vinculan la tabla de hechos `hecho_novedad` con sus respectivas dimensiones. Estas relaciones permiten integrar eficientemente los eventos de novedad con su contexto asociado.

### Relaciones establecidas:

1. **Dimensiones principales**:
   - `key_dim_sede` → `dim_sede(key_dim_sede)`
   - `key_dim_cliente` → `dim_cliente(key_dim_cliente)`
   - `key_dim_mensajero` → `dim_mensajero(key_dim_mensajero)`

2. **Dimensión temporal**:
   - `key_novedad_fecha` → `dim_fecha(key_dim_fecha)`
   - `key_novedad_hora` → `dim_hora(key_dim_hora)`

Estas relaciones permiten, por ejemplo:
- Analizar las novedades por día y hora.
- Evaluar la frecuencia de novedades por sede, cliente o mensajero.
- Estudiar los patrones temporales de fallas o eventos en el proceso de mensajería.

### Implementación técnica:

- Se utilizó `psycopg2` para conectarse a la bodega de datos y ejecutar sentencias SQL `ALTER TABLE`.
- Cada restricción se agrega mediante la instrucción `ADD CONSTRAINT`.
- Se incluye un manejo de errores para evitar fallos en caso de que alguna restricción ya haya sido definida previamente.
- Finalmente, se realiza `commit()` y se cierran las conexiones.

Con estas claves foráneas, se asegura la integridad referencial del modelo y se facilita el análisis multidimensional sobre las novedades registradas.


In [113]:
import psycopg2

conn = psycopg2.connect(
    dbname=dwname,
    user=user,
    password=password,
    host=host,
    port=port
)
cur = conn.cursor()

try:
    # FK a dimensiones: sede, cliente, mensajero
    cur.execute("""
        ALTER TABLE hecho_novedad
        ADD CONSTRAINT fk_sede FOREIGN KEY (key_dim_sede) REFERENCES dim_sede(key_dim_sede),
        ADD CONSTRAINT fk_cliente FOREIGN KEY (key_dim_cliente) REFERENCES dim_cliente(key_dim_cliente),
        ADD CONSTRAINT fk_mensajero FOREIGN KEY (key_dim_mensajero) REFERENCES dim_mensajero(key_dim_mensajero);
    """)

    # FK a dimensión fecha
    for estado in ['novedad']:
        cur.execute(f"""
            ALTER TABLE hecho_novedad
            ADD CONSTRAINT fk_{estado}_fecha FOREIGN KEY (key_{estado}_fecha) REFERENCES dim_fecha(key_dim_fecha);
        """)
    
    # FK a dimensión hora
    for estado in ['novedad']:
        cur.execute(f"""
            ALTER TABLE hecho_novedad
            ADD CONSTRAINT fk_{estado}_hora FOREIGN KEY (key_{estado}_hora) REFERENCES dim_hora(key_dim_hora);
        """)
    
except psycopg2.errors.DuplicateObject as e:
    print("Una o más llaves foráneas ya existen:", e)
finally:
    conn.commit()
    cur.close()
    conn.close()
